# 01 - RRAO Classification and Exclusions

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [RRAO package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: one RRAO positions table with deterministic position identifiers, legal-entity and desk lineage, gross effective notional, evidence type, product descriptors, and exclusion evidence fields. The machine-readable contract is [`frtb_rrao.positions.schema.json`](../../../docs/schemas/input_table/frtb_rrao.positions.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

This notebook walks every position in the `rrao_sample_book_v1` fixture through
the classification tree and shows the cited regulatory decision for each.

Regulatory anchors: Basel MAR23.2–MAR23.7; U.S. NPR 2.0 proposed sections
`__.211(a)` and `__.211(b)`; EU CRR3 Article 325u (comparison context).

Demonstration caution: all positions are synthetic deterministic fixture data.
Classifications shown here are synthetic demonstration evidence, not regulatory capital determinations.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Markdown, display

from examples.notebook_utils import (
    RRAO_CLASSIFICATION_COLORS,
    setup_notebook_plot_style,
)

from examples.rrao_fixture import (
    load_context,
    load_positions,
    PROFILE_US_NPR,
)
from frtb_rrao import (
    classify_rrao_positions,
    RraoClassification,
    RraoEvidenceType,
)

setup_notebook_plot_style()

ctx = load_context(profile=PROFILE_US_NPR)
positions = load_positions()
decisions = classify_rrao_positions(positions, profile=PROFILE_US_NPR)

print(f"Profile: {ctx.profile.value}")
print(f"Positions: {len(positions)}")


## Public API happy path

This notebook demonstrates the public row-level classification and capital entrypoints before the deeper audit views.


## Classification flow

```mermaid
flowchart LR
    A[Raw RRAO positions] --> B[Top-level classify_rrao_positions]
    B --> C{Exclusion evidence?}
    C -->|yes| D[Excluded line with zero add-on]
    C -->|no| E{Residual-risk evidence type}
    E --> F[EXOTIC at 1.00 percent]
    E --> G[OTHER_RESIDUAL_RISK at 0.10 percent]
    E --> H[SUPERVISOR_DIRECTED at 0.10 percent]
    D --> I[Audit table and coverage charts]
    F --> I
    G --> I
    H --> I
```


In [ ]:
from frtb_rrao import calculate_rrao_capital, classify_rrao_positions

public_decisions = classify_rrao_positions(positions, profile=PROFILE_US_NPR)
public_result = calculate_rrao_capital(positions, context=ctx)
print(f"Classified positions: {len(public_decisions)}")
print(f"Total RRAO capital: {public_result.total_rrao:,.2f}")
print(f"Input hash        : {public_result.input_hash}")
print(f"Profile hash      : {public_result.profile_hash}")


## Implementation details / math verification - Classification Summary

The model applies a decision tree: explicit exclusion path first (listed,
clearable, simple option, back-to-back, and US NPR-specific exclusions),
then investment-fund treatment, then the evidence-type rule map.  The result
for each position is a deterministic `RraoClassificationDecision` with a cited
regulatory paragraph.

Risk weights under both Basel MAR23 and US NPR 2.0:
- **EXOTIC** → 1.00% of gross effective notional
- **OTHER_RESIDUAL_RISK** → 0.10% of gross effective notional
- **SUPERVISOR_DIRECTED** → 0.10% (US NPR only)
- **EXCLUDED** → 0.00%

In [ ]:
def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    sep = " | ".join("---" for _ in headers)
    head = " | ".join(headers)
    return Markdown(f"|{head}|\n|{sep}|\n" + "\n".join(f"|{r}|" for r in ("".join(f"{c} | " for c in row).rstrip(" |") for row in rows)))

CLASS_LABEL = {
    "EXOTIC": "[EXOTIC]",
    "OTHER_RESIDUAL_RISK": "[OTHER]",
    "SUPERVISOR_DIRECTED": "[SUPDIR]",
    "EXCLUDED": "[EXCL]",
    "UNSUPPORTED": "[?]",
}

rows = []
pos_by_id = {p.position_id: p for p in positions}
for decision in decisions:
    pos = pos_by_id[decision.position_id]
    cls_label = CLASS_LABEL.get(decision.classification.value, decision.classification.value)
    excl = decision.exclusion_reason.value if decision.exclusion_reason else ""
    cite = ", ".join(decision.citations[:2])
    rows.append([
        decision.position_id,
        pos.desk_id,
        decision.evidence_type.value,
        cls_label,
        excl,
        cite,
    ])

display(Markdown("| Position | Desk | Evidence Type | Classification | Exclusion Reason | Citations |\n" +
    "|---|---|---|---|---|---|\n" +
    "\n".join(f"| {' | '.join(r)} |" for r in rows)))


## Classification Distribution

This chart shows how many positions land in each classification bucket and the
aggregate gross notional behind each outcome.  The key audit observation is that
the EXCLUDED bucket typically contains the largest aggregate notional — exclusions
are consequential and must be fully evidenced.

Figure: RRAO position count and gross notional by classification for the synthetic sample book.


In [ ]:
from collections import defaultdict

cls_notional: dict[str, float] = defaultdict(float)
cls_count: dict[str, int] = defaultdict(int)
for decision in decisions:
    pos = pos_by_id[decision.position_id]
    cls_notional[decision.classification.value] += pos.gross_effective_notional
    cls_count[decision.classification.value] += 1

cls_order = ["EXOTIC", "OTHER_RESIDUAL_RISK", "SUPERVISOR_DIRECTED", "EXCLUDED"]
palette = RRAO_CLASSIFICATION_COLORS

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

counts = [cls_count.get(c, 0) for c in cls_order]
notionals = [cls_notional.get(c, 0) / 1e6 for c in cls_order]
colors = [palette[c] for c in cls_order]

bars1 = ax1.bar(cls_order, counts, color=colors, alpha=0.86)
for bar, count in zip(bars1, counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             str(count), ha="center", va="bottom", fontweight="bold")
ax1.set_title("Position count by classification")
ax1.set_ylabel("Positions")
ax1.set_xticklabels([c.replace("_", "\n") for c in cls_order])

bars2 = ax2.bar(cls_order, notionals, color=colors, alpha=0.86)
for bar, n in zip(bars2, notionals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f"${n:,.0f}M", ha="center", va="bottom", fontweight="bold", fontsize=9)
ax2.set_title("Gross notional (USD M) by classification")
ax2.set_ylabel("USD millions")
ax2.set_xticklabels([c.replace("_", "\n") for c in cls_order])

fig.suptitle("RRAO Classification Distribution — Sample Book v1 (US NPR 2.0)", fontsize=13)
plt.tight_layout()
plt.show()


## Exclusion Breakdown

Excluded positions carry zero add-on regardless of notional size.  The exclusion
decision must be supported by deterministic evidence (e.g., exchange listing
identifier, CCP clearing certificate, back-to-back match evidence ID).  This
table shows the cited exclusion reason and evidence ID for every excluded position.

Note that the back-to-back weather pair (positions `excl-btb-weather-A` / `B`)
illustrates matched-pair exclusion: both legs must be present and cross-reference
the same `match_group_id`.

In [ ]:
excluded_decisions = [d for d in decisions if d.classification is RraoClassification.EXCLUDED]

print(f"Excluded positions: {len(excluded_decisions)}")
print()

rows_excl = []
for d in excluded_decisions:
    pos = pos_by_id[d.position_id]
    btb_note = ""
    if pos.back_to_back_match is not None:
        btb_note = f" (matched: {pos.back_to_back_match.matched_position_id})"
    rows_excl.append([
        d.position_id,
        d.exclusion_reason.value if d.exclusion_reason else "",
        pos.exclusion_evidence_id or "",
        f"${pos.gross_effective_notional/1e6:,.0f}M",
        btb_note or "",
    ])

display(Markdown(
    "| Position | Exclusion Reason | Evidence ID | Notional | Back-to-Back Note |\n"
    "|---|---|---|---|---|\n"
    + "\n".join(f"| {' | '.join(r)} |" for r in rows_excl)
))


## Evidence Type Coverage

This chart checks that the sample book exercises the full evidence-type space
supported by US NPR 2.0.  Each bar is one evidence type; the colour shows the
resulting classification.

Figure: RRAO evidence-type coverage by position count, coloured by resulting classification.


In [ ]:
from frtb_rrao.reference_data import evidence_rules_for_profile

supported_evidence = {rule.evidence_type for rule in evidence_rules_for_profile(PROFILE_US_NPR)}
covered_evidence = {d.evidence_type for d in decisions if d.classification is not RraoClassification.EXCLUDED}

ev_count: dict[str, int] = defaultdict(int)
ev_cls: dict[str, str] = {}
for d in decisions:
    key = d.evidence_type.value
    ev_count[key] += 1
    ev_cls[key] = d.classification.value

ev_order = sorted(ev_count)
fig, ax = plt.subplots(figsize=(13, 4))
bar_colors = [palette.get(ev_cls.get(e, "EXCLUDED"), "#aaaaaa") for e in ev_order]
bars = ax.barh(ev_order, [ev_count[e] for e in ev_order], color=bar_colors, alpha=0.86)
for bar, ev in zip(bars, ev_order):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{ev_count[ev]} pos  ({ev_cls.get(ev, '?')})",
            va="center", fontsize=8)

legend_patches = [mpatches.Patch(color=palette[c], label=c) for c in cls_order]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)
ax.set_title("Position count by evidence type — US NPR 2.0")
ax.set_xlabel("Positions")
ax.set_xlim(0, max(ev_count.values()) + 2)
plt.tight_layout()
plt.show()

# Audit: which supported evidence types appear in the fixture?
print(f"Supported evidence types covered: {len(covered_evidence)}/{len(supported_evidence)}")
missing = supported_evidence - covered_evidence
if missing:
    print(f"Not exercised in sample book: {[e.value for e in missing]}")


## See also

- [RRAO package journey](../docs/PACKAGE_JOURNEY.md)
- [RRAO dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
